# Importing Library

In [61]:
import tensorflow as tf
import matplotlib.pyplot as plt
import numpy as np
from tensorflow.keras.layers import Input, Dropout, Dense, GlobalAveragePooling2D
from tensorflow.keras.models import Model
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
from tensorflow.keras.metrics import SparseCategoricalAccuracy
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
from tensorflow.keras.layers import RandomFlip, RandomRotation, RandomBrightness

# Importing Dataset

In [62]:
IMG_SIZE = (128, 128)
BATCH_SIZE = 16
SEED = 123
train_ds = tf.keras.utils.image_dataset_from_directory(
    "data/train",
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode='int',
    shuffle=True,
    validation_split=0.2,
    subset="training",
    seed=SEED
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    "data/train",
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode='int',
    shuffle=True,
    validation_split=0.2,
    subset="validation",
    seed=SEED
)

test_ds = tf.keras.utils.image_dataset_from_directory(
    "data/test",
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode='int',
    shuffle=True
)

Found 9650 files belonging to 3 classes.
Using 7720 files for training.
Found 9650 files belonging to 3 classes.
Using 1930 files for validation.
Found 2278 files belonging to 3 classes.


# Data Preprocessing

In [63]:
data_augmentation = tf.keras.Sequential([
    RandomRotation(factor=(-0.025, 0.025)),
    RandomFlip(mode="horizontal"),
    RandomBrightness(factor=0.1)
], name="data_augmentation")

# Model Creation

In [64]:
def build_model(input_shape=IMG_SIZE + (3,)):
    base_model = tf.keras.applications.EfficientNetV2S(
        include_top=False,
        input_shape=input_shape,
        weights="imagenet"
    )
    base_model.trainable = False

    inputs = Input(shape=input_shape)
    x = data_augmentation(inputs) 
    x = base_model(x, training=False)
    x = GlobalAveragePooling2D()(x)
    x = Dropout(0.3)(x)
    outputs = Dense(3, activation="softmax")(x)

    model = Model(inputs, outputs, name="Human_Detection_Model")
    model.compile(
        optimizer=tf.keras.optimizers.Adam(1e-3),
        loss="sparse_categorical_crossentropy",
        metrics=[SparseCategoricalAccuracy(name="accuracy")]
    )
    return model, base_model

model, base_model = build_model()
model.summary()

Model: "Human_Detection_Model"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_30 (InputLayer)     │ (None, 128, 128, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ data_augmentation (Sequential)  │ (None, 128, 128, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ efficientnetv2-s (Functional)   │ (None, 4, 4, 1280)     │    20,331,360 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d_13     │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_13 (Dropout)            │ (None, 1280)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_12 (Dense)                │ (None, 3)              │         3,843 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 20,335,203 (77.57 MB)

 Trainable params: 3,843 (15.01 KB)

 Non-trainable params: 20,331,360 (77.56 MB)

# Training The Model

In [65]:
history1 = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=5
)

Epoch 1/5
483/483 ━━━━━━━━━━━━━━━━━━━━ 50s 70ms/step - accuracy: 0.6196 - loss: 0.8309 - val_accuracy: 0.6845 - val_loss: 0.7357
Epoch 2/5
483/483 ━━━━━━━━━━━━━━━━━━━━ 28s 57ms/step - accuracy: 0.6749 - loss: 0.7367 - val_accuracy: 0.6943 - val_loss: 0.7123
Epoch 3/5
483/483 ━━━━━━━━━━━━━━━━━━━━ 27s 55ms/step - accuracy: 0.6968 - loss: 0.7169 - val_accuracy: 0.7067 - val_loss: 0.6945
Epoch 4/5
483/483 ━━━━━━━━━━━━━━━━━━━━ 25s 51ms/step - accuracy: 0.6962 - loss: 0.7024 - val_accuracy: 0.7130 - val_loss: 0.6758
Epoch 5/5
483/483 ━━━━━━━━━━━━━━━━━━━━ 28s 59ms/step - accuracy: 0.7005 - loss: 0.6967 - val_accuracy: 0.7130 - val_loss: 0.6703


# Fine Tuning

In [66]:
print("Base model has", len(base_model.layers), "layers.")
fine_tune_at = len(base_model.layers) - 50 

for layer in base_model.layers[:fine_tune_at]:
    layer.trainable = False
for layer in base_model.layers[fine_tune_at:]:
    layer.trainable = True

model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-5),
    loss="sparse_categorical_crossentropy",
    metrics=[SparseCategoricalAccuracy(name="accuracy")]
)

history2 = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=10,
    callbacks=[
        EarlyStopping(monitor="val_loss", patience=3, restore_best_weights=True),
        ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=2, min_lr=1e-7)
    ]
)

Base model has 513 layers.
Epoch 1/10


E0000 00:00:1757172556.091242   11378 meta_optimizer.cc:967] layout failed: INVALID_ARGUMENT: Size of values 0 does not match size of permutation 4 @ fanin shape inStatefulPartitionedCall/Human_Detection_Model_1/efficientnetv2-s_1/block1b_drop_1/stateless_dropout/SelectV2-2-TransposeNHWCToNCHW-LayoutOptimizer


483/483 ━━━━━━━━━━━━━━━━━━━━ 59s 84ms/step - accuracy: 0.6876 - loss: 0.7501 - val_accuracy: 0.7083 - val_loss: 0.6804 - learning_rate: 1.0000e-05
Epoch 2/10
483/483 ━━━━━━━━━━━━━━━━━━━━ 37s 76ms/step - accuracy: 0.7058 - loss: 0.7034 - val_accuracy: 0.7259 - val_loss: 0.6366 - learning_rate: 1.0000e-05
Epoch 3/10
483/483 ━━━━━━━━━━━━━━━━━━━━ 37s 76ms/step - accuracy: 0.7196 - loss: 0.6731 - val_accuracy: 0.7378 - val_loss: 0.6130 - learning_rate: 1.0000e-05
Epoch 4/10
483/483 ━━━━━━━━━━━━━━━━━━━━ 38s 78ms/step - accuracy: 0.7342 - loss: 0.6362 - val_accuracy: 0.7435 - val_loss: 0.5942 - learning_rate: 1.0000e-05
Epoch 5/10
483/483 ━━━━━━━━━━━━━━━━━━━━ 38s 78ms/step - accuracy: 0.7439 - loss: 0.6162 - val_accuracy: 0.7513 - val_loss: 0.5800 - learning_rate: 1.0000e-05
Epoch 6/10
483/483 ━━━━━━━━━━━━━━━━━━━━ 37s 76ms/step - accuracy: 0.7439 - loss: 0.6153 - val_accuracy: 0.7554 - val_loss: 0.5667 - learning_rate: 1.0000e-05
Epoch 7/10
483/483 ━━━━━━━━━━━━━━━━━━━━ 37s 76ms/step - accurac